# Final Model Selection

This notebook selects a model using validation data only. The test holdout is deliberately excluded from this decision and must be used only once after the model choice is final.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent
if not (ROOT / 'data' / 'raw' / 'wind_turbine_detection.csv').exists():
    ROOT = Path.cwd() / 'PROJECT 1'
METRICS_PATH = ROOT / 'outputs' / 'model_selection' / 'validation_model_metrics.csv'
if not METRICS_PATH.exists():
    raise FileNotFoundError('Run the final export cell in 07_hyperparameter_tuning.ipynb before running this notebook. No test data is needed.')
metrics = pd.read_csv(METRICS_PATH)
required_columns = {'model', 'model_type', 'dataset', 'accuracy', 'precision', 'recall', 'f1'}
missing_columns = required_columns.difference(metrics.columns)
if missing_columns:
    raise ValueError(f'Metrics export is missing: {sorted(missing_columns)}')
display(Markdown(f'Loaded **{len(metrics):,} validation-workflow metric rows**. No test-dataset metrics are loaded or used.'))

In [ ]:
# Full comparison table: each row identifies baseline/tuned status and split.
display(Markdown('## All baseline and tuned model performance'))
comparison_table = metrics[['model', 'model_type', 'dataset', 'accuracy', 'precision', 'recall', 'f1']].sort_values(['model', 'model_type', 'dataset'])
display(comparison_table.style.format({'accuracy': '{:.3f}', 'precision': '{:.3f}', 'recall': '{:.3f}', 'f1': '{:.3f}'}))

training = metrics.loc[metrics.dataset.eq('training')].set_index(['model', 'model_type'])[['accuracy', 'precision', 'recall', 'f1']].add_prefix('training_')
validation = metrics.loc[metrics.dataset.eq('validation')].set_index(['model', 'model_type'])[['accuracy', 'precision', 'recall', 'f1']].add_prefix('validation_')
generalization = validation.join(training, how='inner').reset_index()
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    generalization[f'{metric}_gap_train_minus_validation'] = generalization[f'training_{metric}'] - generalization[f'validation_{metric}']
display(Markdown('## Training versus validation performance'))
display(generalization.style.format({column: '{:.3f}' for column in generalization.select_dtypes('number').columns}))

In [ ]:
# Decision framework: recall first, with safeguards for false alarms and generalisation.
candidates = generalization.copy()
# Tuned models are preferred where they provide equal/better validation performance.
tuned_candidates = candidates[candidates.model_type.eq('Tuned')].copy()
selection_pool = tuned_candidates if not tuned_candidates.empty else candidates.copy()

# Precision and F1 safeguards prevent a recall-only choice that could overwhelm operations.
precision_floor = selection_pool.validation_precision.quantile(.25)
f1_floor = selection_pool.validation_f1.quantile(.25)
reasonable_gap = selection_pool.recall_gap_train_minus_validation.abs().le(.15)
operationally_viable = selection_pool[(selection_pool.validation_precision >= precision_floor) & (selection_pool.validation_f1 >= f1_floor) & reasonable_gap].copy()
if operationally_viable.empty:
    operationally_viable = selection_pool.copy()
    viability_note = 'No model satisfied every screening guardrail, so ranking uses all tuned candidates and flags the trade-off for review.'
else:
    viability_note = 'Candidates below the lower-quartile validation precision/F1 thresholds or with a recall gap above 0.15 were excluded before ranking.'

# Recall dominates the decision; F1 and precision protect practical alert volume; gap rewards generalisation.
operationally_viable['selection_score'] = (0.55 * operationally_viable.validation_recall + 0.25 * operationally_viable.validation_f1 + 0.15 * operationally_viable.validation_precision - 0.05 * operationally_viable.recall_gap_train_minus_validation.abs())
ranked_candidates = operationally_viable.sort_values(['selection_score', 'validation_recall', 'validation_f1', 'validation_precision'], ascending=False).reset_index(drop=True)
display(Markdown('## Validation-based selection ranking'))
display(ranked_candidates.style.format({column: '{:.3f}' for column in ranked_candidates.select_dtypes('number').columns}))

winner = ranked_candidates.iloc[0]
display(Markdown('## Final-model rationale'))
rationale = f"""**Selected model: {winner['model']} ({winner['model_type']}).**

- Validation Recall is **{winner['validation_recall']:.3f}**, the primary measure of successfully detecting true drivetrain faults.
- Validation Precision is **{winner['validation_precision']:.3f}** and validation F1 is **{winner['validation_f1']:.3f}**, so the choice considers false-alarm burden and the Recall–Precision balance rather than Accuracy alone.
- The training-to-validation Recall gap is **{winner['recall_gap_train_minus_validation']:+.3f}**, providing the generalisation check; smaller gaps are preferred.
- The model is suitable for operations-centre use only as a decision-support signal: route alerts for analyst review, monitor alert volume and Recall/Precision drift by turbine, and retain human escalation procedures.
- This is a validation-only choice. The untouched test set must be evaluated once after this selection is approved.

{viability_note}"""
display(Markdown(rationale))